In [1]:
import os
import time
import hashlib
import sqlite3
import re
from datetime import datetime
from collections import deque
from urllib.parse import urlparse, urldefrag, urljoin
from contextlib import contextmanager

import requests
from bs4 import BeautifulSoup
import pandas as pd

print("Nạp thư viện thành công!")

Nạp thư viện thành công!


In [2]:
TOPIC = "Biotechnology & Applied Biology"
SEED_URLS = [
    "https://www.ejbiotechnology.info/index.php/ejbiotechnology/issue/archive"
]

ALLOWED_DOMAINS = [
    "www.ejbiotechnology.info"
]

MAX_DEPTH = 3
MAX_PAGES = 300
REQUEST_TIMEOUT = 15
CRAWL_DELAY = 1.0

DATA_DIR = "data"
DB_PATH = os.path.join(DATA_DIR, "ejbiotech_crawler.db")

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9"
}

IGNORED_EXTENSIONS = (
    ".jpg", ".jpeg", ".png", ".gif", ".svg", ".webp", ".ico",
    ".css", ".js", ".zip", ".tar", ".gz", ".pdf", ".mp4", ".mp3",
    ".exe", ".bin", ".xml", ".tsv", ".csv"
)

print("=" * 25 + " CRAWLER CONFIGURATION " + "=" * 25)
print(f"Topic          : {TOPIC}")
print(f"Seed URLs      : {len(SEED_URLS)}")
for s in SEED_URLS:
    print(f"  - {s}")
print(f"Allowed Domains: {ALLOWED_DOMAINS}")
print(f"Maximum Depth  : {MAX_DEPTH}")
print(f"Maximum Pages  : {MAX_PAGES}")
print(f"Request Timeout: {REQUEST_TIMEOUT}s | Crawl Delay: {CRAWL_DELAY}s")
print("=" * 73)

========================= CRAWLER CONFIGURATION =========================
Topic          : Biotechnology & Applied Biology
Seed URLs      : 1
  - https://www.ejbiotechnology.info/index.php/ejbiotechnology/issue/archive
Allowed Domains: ['www.ejbiotechnology.info']
Maximum Depth  : 3
Maximum Pages  : 300
Request Timeout: 15s | Crawl Delay: 1.0s


In [3]:
class URLFrontier:
    def __init__(self, max_depth=3):
        self.queue = deque()
        self.visited = set()
        self.in_frontier = set()
        self.max_depth = max_depth
        self.discovered_count = 0
        self.skipped_count = 0

    @staticmethod
    def canonicalize_url(raw_url: str) -> str:
        url_no_frag, _ = urldefrag(raw_url)
        parsed = urlparse(url_no_frag)
        scheme = parsed.scheme.lower()
        netloc = parsed.netloc.lower()
        path = parsed.path.rstrip("/")
        
        if not path:
            path = "/"
        return f"{scheme}://{netloc}{path}"

    def add_url(self, url: str, depth: int) -> bool:
        norm_url = self.canonicalize_url(url)
        self.discovered_count += 1

        if depth > self.max_depth:
            self.skipped_count += 1
            return False

        if norm_url in self.visited or norm_url in self.in_frontier:
            self.skipped_count += 1
            return False

        self.in_frontier.add(norm_url)
        self.queue.append((norm_url, depth))
        return True

    def get_next(self):
        if not self.queue:
            return None, None
        url, depth = self.queue.popleft()
        self.in_frontier.discard(url)
        self.visited.add(url)
        return url, depth

    def is_empty(self) -> bool:
        return len(self.queue) == 0

class DuplicateDetector:
    def __init__(self):
        self.fingerprints = set()

    def is_duplicate(self, text_content: str) -> bool:
        if not text_content:
            return True
        fp = hashlib.sha256(text_content.strip().encode("utf-8")).hexdigest()
        if fp in self.fingerprints:
            return True
        self.fingerprints.add(fp)
        return False

frontier = URLFrontier(max_depth=MAX_DEPTH)
detector = DuplicateDetector()
print("Đã khởi tạo URL Frontier và Duplicate Detector.")

Đã khởi tạo URL Frontier và Duplicate Detector.


In [4]:
class Database:
    def __init__(self, db_path: str):
        self.db_path = db_path
        os.makedirs(os.path.dirname(db_path), exist_ok=True)
        self.create_tables()

    @contextmanager
    def get_cursor(self):
        conn = sqlite3.connect(self.db_path, timeout=20.0)
        cursor = conn.cursor()
        try:
            yield cursor
            conn.commit()
        finally:
            conn.close()

    def create_tables(self):
        with self.get_cursor() as cursor:
            cursor.execute('''
                CREATE TABLE IF NOT EXISTS pages (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    url TEXT UNIQUE,
                    domain TEXT,
                    title TEXT,
                    abstract TEXT,
                    authors TEXT,
                    content TEXT,
                    is_article INTEGER DEFAULT 0,
                    depth INTEGER,
                    status_code INTEGER,
                    crawled_at TEXT
                )
            ''')

            cursor.execute('''
                CREATE TABLE IF NOT EXISTS links (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    source_url TEXT,
                    target_url TEXT
                )
            ''')

            cursor.execute('CREATE INDEX IF NOT EXISTS idx_pages_url ON pages(url)')
            cursor.execute('CREATE INDEX IF NOT EXISTS idx_links_source ON links(source_url)')

    def insert_page(self, url, domain, title, abstract, authors, content, is_article, depth, status_code, crawled_at):
        with self.get_cursor() as cursor:
            cursor.execute('''
                INSERT OR IGNORE INTO pages (url, domain, title, abstract, authors, content, is_article, depth, status_code, crawled_at)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ''', (url, domain, title, abstract, authors, content, is_article, depth, status_code, crawled_at))

    def insert_links_batch(self, link_tuples):
        if not link_tuples:
            return
        with self.get_cursor() as cursor:
            cursor.executemany('''
                INSERT INTO links (source_url, target_url)
                VALUES (?, ?)
            ''', link_tuples)

db = Database(DB_PATH)
print("Đã kết nối và khởi tạo SQLite Database chuẩn.")

Đã kết nối và khởi tạo SQLite Database chuẩn.


In [5]:
ARTICLE_REGEX = re.compile(r"/index\.php/ejbiotechnology/article/view/[^/]+$", re.IGNORECASE)

def is_article_url(url: str) -> bool:
    return bool(ARTICLE_REGEX.search(url))

def is_allowed_navigation_url(url: str) -> bool:
    parsed = urlparse(url)
    if parsed.netloc.lower() not in ALLOWED_DOMAINS:
        return False
    path = parsed.path.lower()

    blacklists = ["/login", "/user/register", "/about", "/search", "/gateway/", "/feedback"]
    if any(b in path for b in blacklists):
        return False

    return "/issue/archive" in path or "/issue/view" in path

def extract_page_data(html_content: str, current_url: str) -> tuple[bool, dict, BeautifulSoup]:
    soup = BeautifulSoup(html_content, "html.parser")

    # Làm sạch văn bản toàn trang để lấy content thuần
    soup_clean = BeautifulSoup(html_content, "html.parser")
    for tag in soup_clean(["script", "style", "noscript", "nav", "footer", "header", "aside", "svg"]):
        tag.decompose()
    clean_text = soup_clean.get_text(separator=" ", strip=True)

    # 1. KIỂM TRA ĐIỀU KIỆN LÀ BÀI BÁO (Dựa vào URL):
    # Nếu không phải URL dạng /article/view/{id} (ví dụ /issue/archive hoặc /issue/view/...) -> Đều là NAV PAGE
    if not is_article_url(current_url):
        title = soup.title.get_text(strip=True) if soup.title else "Navigation Page"
        return False, {
            "title": title,
            "abstract": "",
            "authors": "",
            "content": clean_text,
            "is_article": 0
        }, soup

    # 2. TRÍCH XUẤT BÀI BÁO THẬT SỰ (Depth 2):
    # --- A. TIÊU ĐỀ (Trích xuất từ thẻ <h1> trên giao diện) ---
    title_el = (
        soup.find("h1", class_=re.compile(r"page_title|title|article-details-title", re.I))
        or soup.find("h1")
    )
    if title_el:
        title = title_el.get_text(strip=True)
    else:
        # Fallback duy nhất cho meta: chỉ dùng citation_title (không dùng dc.title)
        meta_title = soup.find("meta", attrs={"name": re.compile(r"^citation_title$", re.I)})
        title = meta_title.get("content", "").strip() if meta_title else (soup.title.get_text(strip=True) if soup.title else "No Title")

    # --- B. TÁC GIẢ (Trích xuất từ danh sách tác giả hiển thị trên giao diện) ---
    authors_list = []
    author_nodes = soup.find_all(class_=re.compile(r"author-name|authors?_name|item authors?|name", re.I))
    for node in author_nodes:
        txt = node.get_text(strip=True)
        if txt and len(txt) < 80 and not re.search(r"department|hospital|university|china|vietnam|valparaíso", txt, re.I):
            if txt not in authors_list:
                authors_list.append(txt)

    # Fallback chỉ dùng citation_author (loại trừ hẳn dc.creator)
    if not authors_list:
        meta_authors = soup.find_all("meta", attrs={"name": re.compile(r"^citation_author$", re.I)})
        authors_list = [m["content"].strip() for m in meta_authors if m.get("content")]

    authors_str = ", ".join(authors_list) if authors_list else "Unknown"

    # --- C. TÓM TẮT (Trích xuất khối Abstract trên giao diện) ---
    abstract = ""
    abs_block = soup.find(["div", "section"], class_=re.compile(r"item abstract|article-abstract|abstract", re.I))
    if abs_block:
        for heading in abs_block.find_all(["h2", "h3", "h4", "strong"]):
            if "abstract" in heading.get_text(strip=True).lower():
                heading.decompose()
        abstract = abs_block.get_text(separator=" ", strip=True)
    else:
        meta_abs = soup.find("meta", attrs={"name": re.compile(r"^citation_abstract$", re.I)})
        if meta_abs and meta_abs.get("content"):
            abstract = meta_abs["content"].strip()

    return True, {
        "title": title,
        "abstract": abstract,
        "authors": authors_str,
        "content": clean_text,
        "is_article": 1
    }, soup

def extract_and_filter_links(soup: BeautifulSoup, base_url: str) -> list[str]:
    valid_links = []
    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        if href.startswith(("mailto:", "javascript:", "tel:", "#")):
            continue

        full_url = urljoin(base_url, href)
        parsed = urlparse(full_url)

        if parsed.scheme not in ("http", "https"):
            continue
        if parsed.netloc.lower() not in ALLOWED_DOMAINS:
            continue
        if any(parsed.path.lower().endswith(ext) for ext in IGNORED_EXTENSIONS):
            continue

        clean_url = URLFrontier.canonicalize_url(full_url)
        # Chỉ giữ lại liên kết điều hướng (archive, issue) hoặc liên kết bài viết chi tiết
        if is_article_url(clean_url) or is_allowed_navigation_url(clean_url):
            valid_links.append(clean_url)

    return list(set(valid_links))

In [6]:
article_count = 0
failed_requests = 0
status_code_stats = {}
depth_stats = {d: 0 for d in range(MAX_DEPTH + 1)}
visited_urls = set()
visited_articles = set()

for seed in SEED_URLS:
    frontier.add_url(seed, depth=0)

session = requests.Session()
session.headers.update(HEADERS)

print("=== BẮT ĐẦU CRAWL EJBIOTECHNOLOGY ===")

while not frontier.is_empty() and article_count < MAX_PAGES:
    current_url, current_depth = frontier.get_next()
    if not current_url:
        break

    canonical_url = URLFrontier.canonicalize_url(current_url)
    if canonical_url in visited_urls or canonical_url in visited_articles:
        continue

    start_time = time.time()
    try:
        response = session.get(canonical_url, timeout=REQUEST_TIMEOUT)
        elapsed = time.time() - start_time
        status_code = response.status_code
    except requests.RequestException as e:
        failed_requests += 1
        print(f"[Error] {canonical_url}: {e}")
        continue

    status_code_stats[status_code] = status_code_stats.get(status_code, 0) + 1

    if response.status_code != 200 or "text/html" not in response.headers.get("Content-Type", ""):
        time.sleep(CRAWL_DELAY)
        continue

    visited_urls.add(canonical_url)
    response.encoding = "utf-8"

    is_article, page_data, soup = extract_page_data(response.text, canonical_url)
    crawled_at = datetime.now().isoformat()
    domain = urlparse(canonical_url).netloc

    raw_links = extract_and_filter_links(soup, canonical_url)
    canonical_links = list(set([URLFrontier.canonicalize_url(l) for l in raw_links]))

    link_tuples = [(canonical_url, l) for l in canonical_links]
    db.insert_links_batch(link_tuples)

    if current_depth < MAX_DEPTH:
        for link in canonical_links:
            if link not in visited_urls and link not in visited_articles:
                frontier.add_url(link, depth=current_depth + 1)

  
    if is_article:
        if detector.is_duplicate(page_data["content"]):
            print(f"[Duplicate Skip] {canonical_url}")
            continue

        visited_articles.add(canonical_url)
        article_count += 1
        depth_stats[current_depth] = depth_stats.get(current_depth, 0) + 1

        db.insert_page(
            url=canonical_url,
            domain=domain,
            title=page_data["title"],
            abstract=page_data["abstract"],
            authors=page_data["authors"],
            content=page_data["content"],
            is_article=1,
            depth=current_depth,
            status_code=status_code,
            crawled_at=crawled_at
        )
        print(f"[{article_count:03d}] [ARTICLE] Depth: {current_depth} | Links: {len(canonical_links)} | Time: {elapsed:.2f}s | {page_data['title'][:38]}... | {canonical_url}")
    else:
        db.insert_page(
            url=canonical_url,
            domain=domain,
            title=page_data["title"],
            abstract="",
            authors="",
            content=page_data["content"],
            is_article=0,
            depth=current_depth,
            status_code=status_code,
            crawled_at=crawled_at
        )
        print(f"[NAV PAGE] Depth: {current_depth} | Links: {len(canonical_links)} | {canonical_url}")

    time.sleep(CRAWL_DELAY)


print("\n" + "=" * 25 + " CRAWLING SUMMARY " + "=" * 25)
print(f"Topic Focus            : {TOPIC}")
print(f"Seed URLs              : {len(SEED_URLS)}")
print(f"Scientific Articles    : {article_count} / {MAX_PAGES} (Target)")
print(f"Unique URLs Discovered : {frontier.discovered_count}")
print(f"Skipped URLs           : {frontier.skipped_count}")
print(f"Failed Requests        : {failed_requests}")
print("\n--- Distribution by Depth ---")
for d in range(MAX_DEPTH + 1):
    print(f"  - Depth {d}: {depth_stats.get(d, 0)} articles")
print("\n--- HTTP Status Codes ---")
for code, cnt in sorted(status_code_stats.items()):
    print(f"  - HTTP {code}: {cnt}")
print("=" * 68)

=== BẮT ĐẦU CRAWL EJBIOTECHNOLOGY ===
[NAV PAGE] Depth: 0 | Links: 52 | https://www.ejbiotechnology.info/index.php/ejbiotechnology/issue/archive
[NAV PAGE] Depth: 1 | Links: 5 | https://www.ejbiotechnology.info/index.php/ejbiotechnology/issue/view/167
[NAV PAGE] Depth: 1 | Links: 7 | https://www.ejbiotechnology.info/index.php/ejbiotechnology/issue/view/148
[NAV PAGE] Depth: 1 | Links: 9 | https://www.ejbiotechnology.info/index.php/ejbiotechnology/issue/view/170
[NAV PAGE] Depth: 1 | Links: 14 | https://www.ejbiotechnology.info/index.php/ejbiotechnology/issue/view/124
[Error] https://www.ejbiotechnology.info/index.php/ejbiotechnology/issue/view/166: HTTPSConnectionPool(host='www.ejbiotechnology.info', port=443): Read timed out.
[NAV PAGE] Depth: 1 | Links: 7 | https://www.ejbiotechnology.info/index.php/ejbiotechnology/issue/view/153
[NAV PAGE] Depth: 1 | Links: 10 | https://www.ejbiotechnology.info/index.php/ejbiotechnology/issue/view/146
[NAV PAGE] Depth: 1 | Links: 12 | https://www.ej

In [7]:
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute("SELECT COUNT(*) FROM pages")
total_pages = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM pages WHERE is_article = 1")
total_articles = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM links")
total_links = cursor.fetchone()[0]

print(f"-> Tổng số trang trong DB (tính cả nav): {total_pages}")
print(f"-> Tổng số BÀI BÁO KHOA HỌC chuẩn (is_article = 1): {total_articles}")
print(f"-> Tổng số liên kết trong bảng 'links': {total_links}\n")

if total_articles > 0:
    df = pd.read_sql_query(
        "SELECT id, title, authors, abstract, url, depth, crawled_at FROM pages WHERE is_article = 1 LIMIT 5", 
        conn
    )
    df['abstract'] = df['abstract'].apply(lambda x: x[:100] + '...' if len(x) > 100 else x)
    display(df)

conn.close()
print("Đã đóng kết nối SQLite an toàn.")

-> Tổng số trang trong DB (tính cả nav): 399
-> Tổng số BÀI BÁO KHOA HỌC chuẩn (is_article = 1): 300
-> Tổng số liên kết trong bảng 'links': 2920



,id,title,authors,abstract,url,depth,crawled_at
0,49,AKT/p65-dependent upregulation of CD64 by LPS ...,"Lin Li, Xiaoqing Fu, Nayun Chen, Daihua Fang",Background: Sepsis is a life-threatening condi...,https://www.ejbiotechnology.info/index.php/ejb...,2,2026-10-07T06:48:46.247701
1,50,Exploring the molecular mechanism of resveratr...,"Nan Chen, Yang Yang, Zhuoyu Chen, Huanzhen F...",Background: Resveratrol (RES) has been found t...,https://www.ejbiotechnology.info/index.php/ejb...,2,2026-10-07T06:48:55.496541
2,51,Systems-level analysis prioritizes the importa...,"Khawar Juweria, Jinlei Guo, Saeed Akhtar, Baog...",Background: Head and neck cancer (HNC) is one ...,https://www.ejbiotechnology.info/index.php/ejb...,2,2026-10-07T06:49:08.679946
3,52,High-level expression of an acetaldehyde dehyd...,"Zhe Wang, Jian-Ju He, Xin-Xin Liu, Hong-Ling S...",Background: Acetaldehyde dehydrogenase (ALDH) ...,https://www.ejbiotechnology.info/index.php/ejb...,2,2026-10-07T06:49:54.912352
4,53,LOXL2 promotes tumor proliferation and metasta...,"Yingmin Liu, Xinya Liu, Fei Chen, Wei Nian, Xi...",Background: Esophageal squamous cell carcinoma...,https://www.ejbiotechnology.info/index.php/ejb...,2,2026-10-07T06:50:12.065305


Đã đóng kết nối SQLite an toàn.
